In [ ]:
import torch
import pandas as pd
import numpy as np
from src.metrics import evaluate, confusion, save_preds
from src.data import load_splits
import sys
import os
from pathlib import Path
if "COLAB_GPU" in os.environ or "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    # <- adjust if your folder is elsewhere
    ROOT = Path("/content/drive/MyDrive/labr-rating")
else:
    ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
DATA, PRED, RES = ROOT / "data", ROOT / "predictions", ROOT / "results"

print("GPU:", torch.cuda.is_available())

In [ ]:
from sklearn.model_selection import train_test_split
val = load_splits(DATA)["val"]
sub, _ = train_test_split(val, train_size=1000, stratify=val.rating,
                          random_state=42)  # same proportions as val
y = sub.rating.values
sub.rating.value_counts().sort_index()

In [ ]:
from transformers import pipeline
zs = pipeline("zero-shot-classification",
              model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
              device=0 if torch.cuda.is_available() else -1)   # 0 = first GPU, -1 = CPU

In [ ]:
labels = ["one star", "two stars", "three stars", "four stars", "five stars"]
out = zs(sub.text.iloc[0], candidate_labels=labels,
         hypothesis_template="The reader rated this book {}.")   # {} is replaced by each label -> 5 hypotheses
print("true:", y[0])
print(sub.text.iloc[0][:200])
dict(zip(out["labels"], np.round(out["scores"], 3)))

In [ ]:
from tqdm.auto import tqdm


def run_zs(texts, labels, template, bs=16):
    P = []
    for out in tqdm(zs(list(texts), candidate_labels=labels, hypothesis_template=template,
                       batch_size=bs), total=len(texts)):
        # the pipeline sorts by score -> we reorder
        s = dict(zip(out["labels"], out["scores"]))
        # back to star order 1..5
        P.append([s[l] for l in labels])
    return np.array(P)


def preds_from(P):
    # column 0 = 1 star
    argmax = P.argmax(1) + 1
    expected = np.clip(np.rint(P @ np.arange(1, 6)), 1,
                       5).astype(int)  # weighted average of stars
    return argmax, expected

In [ ]:
variants = {
    "en_stars": (["one star", "two stars", "three stars", "four stars", "five stars"],
                 "The reader rated this book {}."),
    "en_sentiment": (["very negative", "negative", "mixed", "positive", "very positive"],
                     "The opinion of this review is {}."),
    "ar_stars": (["نجمة واحدة", "نجمتين", "ثلاث نجوم", "أربع نجوم", "خمس نجوم"],
                 "قيم القارئ هذا الكتاب ب{}."),
}
results, probs = [], {}
for name, (labs, tpl) in variants.items():
    probs[name] = P = run_zs(sub.text, labs, tpl)
    a, e = preds_from(P)
    results += [evaluate(y, a, f"B_{name}_argmax"),
                evaluate(y, e, f"B_{name}_expected")]
pd.DataFrame(results)

In [ ]:
# A3 predictions for exactly these rows
a3 = pd.read_csv(PRED / "A3_tfidf_LR.csv").set_index("row").loc[sub.row]
results.append(evaluate(a3.y_true, a3.y_pred, "A3_tfidf_LR (same 1000)"))
res = pd.DataFrame(results)
res.to_csv(RES / "phaseB_val1000.csv", index=False)
res

In [ ]:
best = res.iloc[:-1].sort_values("macro_f1").iloc[-1].model   # best B row
print(best)
name = best.replace("B_", "").rsplit("_", 1)[0]
a, e = preds_from(probs[name])
p = a if best.endswith("argmax") else e
save_preds(sub, p, "B_zeroshot_best", PRED)
confusion(y, p)